In [1]:
import pandas as pd

df = pd.read_csv("hybrid_manufacturing_categorical.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

Shape: (1000, 13)

Columns:
['Job_ID', 'Machine_ID', 'Operation_Type', 'Material_Used', 'Processing_Time', 'Energy_Consumption', 'Machine_Availability', 'Scheduled_Start', 'Scheduled_End', 'Actual_Start', 'Actual_End', 'Job_Status', 'Optimization_Category']

First 5 rows:


,Job_ID,Machine_ID,Operation_Type,Material_Used,Processing_Time,Energy_Consumption,Machine_Availability,Scheduled_Start,Scheduled_End,Actual_Start,Actual_End,Job_Status,Optimization_Category
0,J001,M01,Grinding,3.17,76,11.42,96,2023-03-18 08:00:00,2023-03-18 09:16:00,2023-03-18 08:05:00,2023-03-18 09:21:00,Completed,Moderate Efficiency
1,J002,M01,Grinding,3.35,79,6.61,84,2023-03-18 08:10:00,2023-03-18 09:29:00,2023-03-18 08:20:00,2023-03-18 09:39:00,Delayed,Low Efficiency
2,J003,M04,Additive,2.29,56,11.11,92,2023-03-18 08:20:00,2023-03-18 09:16:00,NaN,NaN,Failed,Low Efficiency
3,J004,M04,Grinding,1.76,106,12.50,95,2023-03-18 08:30:00,2023-03-18 10:16:00,2023-03-18 08:35:00,2023-03-18 10:21:00,Completed,Moderate Efficiency
4,J005,M01,Lathe,1.90,46,8.13,88,2023-03-18 08:40:00,2023-03-18 09:26:00,2023-03-18 08:42:00,2023-03-18 09:28:00,Completed,High Efficiency



Data types:
Job_ID                       str
Machine_ID                   str
Operation_Type               str
Material_Used            float64
Processing_Time            int64
Energy_Consumption       float64
Machine_Availability       int64
Scheduled_Start              str
Scheduled_End                str
Actual_Start                 str
Actual_End                   str
Job_Status                   str
Optimization_Category        str
dtype: object

Missing values:
Job_ID                     0
Machine_ID                 0
Operation_Type             0
Material_Used              0
Processing_Time            0
Energy_Consumption         0
Machine_Availability       0
Scheduled_Start            0
Scheduled_End              0
Actual_Start             129
Actual_End               129
Job_Status                 0
Optimization_Category      0
dtype: int64


In [2]:
# Production Insight KPI calculations

import pandas as pd

df = pd.read_csv("hybrid_manufacturing_categorical.csv")

total_jobs = len(df)

completed_jobs = (df["Job_Status"] == "Completed").sum()
delayed_jobs = (df["Job_Status"] == "Delayed").sum()
failed_jobs = (df["Job_Status"] == "Failed").sum()

completion_rate = (completed_jobs / total_jobs) * 100

avg_processing_time = df["Processing_Time"].mean()

total_energy = df["Energy_Consumption"].sum()
energy_per_job = total_energy / total_jobs

avg_machine_availability = df["Machine_Availability"].mean()

efficiency_distribution = df["Optimization_Category"].value_counts()

print("Total Jobs:", total_jobs)
print("Completed:", completed_jobs)
print("Delayed:", delayed_jobs)
print("Failed:", failed_jobs)

print("Completion Rate:", round(completion_rate, 2), "%")
print("Average Processing Time:", round(avg_processing_time, 2), "min")
print("Total Energy:", round(total_energy, 2))
print("Energy per Job:", round(energy_per_job, 2))
print("Average Machine Availability:", round(avg_machine_availability, 2), "%")

print("\nEfficiency Distribution:")
print(efficiency_distribution)

Total Jobs: 1000
Completed: 673
Delayed: 198
Failed: 129
Completion Rate: 67.3 %
Average Processing Time: 71.38 min
Total Energy: 8521.34
Energy per Job: 8.52
Average Machine Availability: 89.16 %

Efficiency Distribution:
Optimization_Category
Low Efficiency         650
Moderate Efficiency    183
High Efficiency        161
Optimal Efficiency       6
Name: count, dtype: int64


In [3]:
X = df[
    [
        "Machine_ID",
        "Operation_Type",
        "Material_Used",
        "Processing_Time",
        "Energy_Consumption",
        "Machine_Availability",
        "Scheduled_Start",
        "Scheduled_End"
    ]
]

y = df["Job_Status"]

print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)


Features:
['Machine_ID', 'Operation_Type', 'Material_Used', 'Processing_Time', 'Energy_Consumption', 'Machine_Availability', 'Scheduled_Start', 'Scheduled_End']

Target:
Job_Status


In [4]:
X["Scheduled_Start"] = pd.to_datetime(X["Scheduled_Start"])
X["Scheduled_End"] = pd.to_datetime(X["Scheduled_End"])

X["Scheduled_Start_Hour"] = X["Scheduled_Start"].dt.hour
X["Scheduled_Start_Minute"] = X["Scheduled_Start"].dt.minute

X["Scheduled_End_Hour"] = X["Scheduled_End"].dt.hour
X["Scheduled_End_Minute"] = X["Scheduled_End"].dt.minute

X = X.drop(columns=["Scheduled_Start", "Scheduled_End"])

X.head()

,Machine_ID,Operation_Type,Material_Used,Processing_Time,Energy_Consumption,Machine_Availability,Scheduled_Start_Hour,Scheduled_Start_Minute,Scheduled_End_Hour,Scheduled_End_Minute
0,M01,Grinding,3.17,76,11.42,96,8,0,9,16
1,M01,Grinding,3.35,79,6.61,84,8,10,9,29
2,M04,Additive,2.29,56,11.11,92,8,20,9,16
3,M04,Grinding,1.76,106,12.50,95,8,30,10,16
4,M01,Lathe,1.90,46,8.13,88,8,40,9,26


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target distribution:")
print(y_train.value_counts())

print("\nTesting target distribution:")
print(y_test.value_counts())

Training data: (800, 10)
Testing data: (200, 10)

Training target distribution:
Job_Status
Completed    539
Delayed      158
Failed       103
Name: count, dtype: int64

Testing target distribution:
Job_Status
Completed    134
Delayed       40
Failed        26
Name: count, dtype: int64


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = [
    "Machine_ID",
    "Operation_Type"
]

numerical_features = [
    "Material_Used",
    "Processing_Time",
    "Energy_Consumption",
    "Machine_Availability",
    "Scheduled_Start_Hour",
    "Scheduled_Start_Minute",
    "Scheduled_End_Hour",
    "Scheduled_End_Minute"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numerical_features
        )
    ]
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                class_weight="balanced"
            )
        )
    ]
)

model.fit(X_train, y_train)

print("Production prediction model trained successfully.")

Production prediction model trained successfully.


In [8]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Make predictions on unseen test data
y_pred = model.predict(X_test)

# Accuracy
accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Model Accuracy: 59.0 %

Classification Report:
              precision    recall  f1-score   support

   Completed       0.67      0.81      0.73       134
     Delayed       0.20      0.12      0.15        40
      Failed       0.38      0.19      0.26        26

    accuracy                           0.59       200
   macro avg       0.42      0.37      0.38       200
weighted avg       0.54      0.59      0.55       200


Confusion Matrix:
[[108  19   7]
 [ 34   5   1]
 [ 20   1   5]]


In [9]:
from sklearn.metrics import balanced_accuracy_score

print("Accuracy:", round(accuracy_score(y_test, y_pred) * 100, 2), "%")
print("Balanced Accuracy:", round(
    balanced_accuracy_score(y_test, y_pred) * 100, 2
), "%")

Accuracy: 59.0 %
Balanced Accuracy: 37.44 %


In [10]:
import pandas as pd

comparison = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

print(comparison.value_counts())

Actual     Predicted
Completed  Completed    108
Delayed    Completed     34
Failed     Completed     20
Completed  Delayed       19
           Failed         7
Delayed    Delayed        5
Failed     Failed         5
Delayed    Failed         1
Failed     Delayed        1
Name: count, dtype: int64


In [11]:
print("Job Status vs Machine ID")
print(pd.crosstab(df["Machine_ID"], df["Job_Status"], normalize="index").round(2))

print("\nJob Status vs Operation Type")
print(pd.crosstab(df["Operation_Type"], df["Job_Status"], normalize="index").round(2))

print("\nAverage numerical features by Job Status")
print(
    df.groupby("Job_Status")[
        [
            "Material_Used",
            "Processing_Time",
            "Energy_Consumption",
            "Machine_Availability"
        ]
    ].mean().round(2)
)

Job Status vs Machine ID
Job_Status  Completed  Delayed  Failed
Machine_ID                            
M01              0.72     0.17    0.11
M02              0.70     0.20    0.10
M03              0.66     0.18    0.16
M04              0.64     0.18    0.18
M05              0.64     0.25    0.11

Job Status vs Operation Type
Job_Status      Completed  Delayed  Failed
Operation_Type                            
Additive             0.71     0.16    0.13
Drilling             0.69     0.20    0.11
Grinding             0.66     0.22    0.12
Lathe                0.65     0.21    0.15
Milling              0.67     0.20    0.13

Average numerical features by Job Status
            Material_Used  Processing_Time  Energy_Consumption  \
Job_Status                                                       
Completed            3.02            71.26                8.52   
Delayed              3.05            71.51                8.47   
Failed               3.00            71.84                8.62   

In [12]:
print("Optimization Category Distribution:")
print(df["Optimization_Category"].value_counts())

print("\nOptimization Category vs Job Status:")
print(
    pd.crosstab(
        df["Optimization_Category"],
        df["Job_Status"],
        normalize="index"
    ).round(2)
)

Optimization Category Distribution:
Optimization_Category
Low Efficiency         650
Moderate Efficiency    183
High Efficiency        161
Optimal Efficiency       6
Name: count, dtype: int64

Optimization Category vs Job Status:
Job_Status             Completed  Delayed  Failed
Optimization_Category                            
High Efficiency              1.0      0.0     0.0
Low Efficiency               0.5      0.3     0.2
Moderate Efficiency          1.0      0.0     0.0
Optimal Efficiency           1.0      0.0     0.0


In [13]:
y_binary = df["Job_Status"].replace({
    "Completed": "Normal",
    "Delayed": "At Risk",
    "Failed": "At Risk"
})

print(y_binary.value_counts())
print("\nClass percentages:")
print((y_binary.value_counts(normalize=True) * 100).round(2))

Job_Status
Normal     673
At Risk    327
Name: count, dtype: int64

Class percentages:
Job_Status
Normal     67.3
At Risk    32.7
Name: proportion, dtype: float64


In [14]:
X_train_b, X_test_b, y_train_b, y_test_b = train_test_split(
    X,
    y_binary,
    test_size=0.2,
    random_state=42,
    stratify=y_binary
)

print("Training data:", X_train_b.shape)
print("Testing data:", X_test_b.shape)

print("\nTraining target distribution:")
print(y_train_b.value_counts())

print("\nTesting target distribution:")
print(y_test_b.value_counts())

Training data: (800, 10)
Testing data: (200, 10)

Training target distribution:
Job_Status
Normal     538
At Risk    262
Name: count, dtype: int64

Testing target distribution:
Job_Status
Normal     135
At Risk     65
Name: count, dtype: int64


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

risk_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight="balanced"
            )
        )
    ]
)

risk_model.fit(X_train_b, y_train_b)

print("Production Risk model trained successfully.")

Production Risk model trained successfully.


In [16]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

y_risk_pred = risk_model.predict(X_test_b)

print("Accuracy:",
      round(accuracy_score(y_test_b, y_risk_pred) * 100, 2), "%")

print("Balanced Accuracy:",
      round(balanced_accuracy_score(y_test_b, y_risk_pred) * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test_b, y_risk_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_b, y_risk_pred))

Accuracy: 63.5 %
Balanced Accuracy: 54.62 %

Classification Report:
              precision    recall  f1-score   support

     At Risk       0.41      0.29      0.34        65
      Normal       0.70      0.80      0.75       135

    accuracy                           0.64       200
   macro avg       0.56      0.55      0.54       200
weighted avg       0.61      0.64      0.62       200


Confusion Matrix:
[[ 19  46]
 [ 27 108]]


In [17]:
print("Energy Consumption statistics by Job Status:\n")

print(
    df.groupby("Job_Status")["Energy_Consumption"]
      .agg(["count", "mean", "std", "min", "max"])
      .round(2)
)

print("\nProcessing Time statistics by Job Status:\n")

print(
    df.groupby("Job_Status")["Processing_Time"]
      .agg(["count", "mean", "std", "min", "max"])
      .round(2)
)

Energy Consumption statistics by Job Status:

            count  mean   std   min    max
Job_Status                                
Completed     673  8.52  3.63  2.01  14.98
Delayed       198  8.47  3.77  2.05  14.90
Failed        129  8.62  3.76  2.12  14.94

Processing Time statistics by Job Status:

            count   mean    std  min  max
Job_Status                               
Completed     673  71.26  28.39   20  120
Delayed       198  71.51  28.98   20  120
Failed        129  71.84  28.41   20  120


In [18]:
print("Optimization Category by Job Status:\n")
print(
    pd.crosstab(
        df["Optimization_Category"],
        df["Job_Status"]
    )
)

print("\nAverage numerical features by Optimization Category:\n")

print(
    df.groupby("Optimization_Category")[
        [
            "Material_Used",
            "Processing_Time",
            "Energy_Consumption",
            "Machine_Availability"
        ]
    ].mean().round(2)
)

Optimization Category by Job Status:

Job_Status             Completed  Delayed  Failed
Optimization_Category                            
High Efficiency              161        0       0
Low Efficiency               323      198     129
Moderate Efficiency          183        0       0
Optimal Efficiency             6        0       0

Average numerical features by Optimization Category:

                       Material_Used  Processing_Time  Energy_Consumption  \
Optimization_Category                                                       
High Efficiency                 2.93            69.35                5.59   
Low Efficiency                  3.03            72.64                8.49   
Moderate Efficiency             3.10            68.13               11.41   
Optimal Efficiency              2.99            89.33                2.46   

                       Machine_Availability  
Optimization_Category                        
High Efficiency                       89.32  
Low Ef

In [19]:
print(
    df.groupby("Optimization_Category")[
        [
            "Energy_Consumption",
            "Processing_Time",
            "Material_Used",
            "Machine_Availability"
        ]
    ].agg(["min", "max", "mean"]).round(2)
)

                      Energy_Consumption               Processing_Time       \
                                     min    max   mean             min  max   
Optimization_Category                                                         
High Efficiency                     2.05   9.65   5.59              20  120   
Low Efficiency                      2.05  14.98   8.49              20  120   
Moderate Efficiency                 7.12  14.98  11.41              21  118   
Optimal Efficiency                  2.01   3.13   2.46              73  112   

                             Material_Used             Machine_Availability  \
                        mean           min   max  mean                  min   
Optimization_Category                                                         
High Efficiency        69.35          1.03  4.97  2.93                   80   
Low Efficiency         72.64          1.01  4.99  3.03                   80   
Moderate Efficiency    68.13          1.01  5.00  3

In [20]:
print(df["Optimization_Category"].value_counts())

print("\nPercentage distribution:")
print(
    (df["Optimization_Category"].value_counts(normalize=True) * 100)
    .round(2)
)

Optimization_Category
Low Efficiency         650
Moderate Efficiency    183
High Efficiency        161
Optimal Efficiency       6
Name: count, dtype: int64

Percentage distribution:
Optimization_Category
Low Efficiency         65.0
Moderate Efficiency    18.3
High Efficiency        16.1
Optimal Efficiency      0.6
Name: proportion, dtype: float64


In [21]:
y_efficiency = df["Optimization_Category"].replace({
    "Optimal Efficiency": "High Efficiency"
})

print(y_efficiency.value_counts())

print("\nPercentage distribution:")
print(
    (y_efficiency.value_counts(normalize=True) * 100)
    .round(2)
)

Optimization_Category
Low Efficiency         650
Moderate Efficiency    183
High Efficiency        167
Name: count, dtype: int64

Percentage distribution:
Optimization_Category
Low Efficiency         65.0
Moderate Efficiency    18.3
High Efficiency        16.7
Name: proportion, dtype: float64


In [22]:
X_train_e, X_test_e, y_train_e, y_test_e = train_test_split(
    X,
    y_efficiency,
    test_size=0.2,
    random_state=42,
    stratify=y_efficiency
)

print("Training data:", X_train_e.shape)
print("Testing data:", X_test_e.shape)

print("\nTraining target distribution:")
print(y_train_e.value_counts())

print("\nTesting target distribution:")
print(y_test_e.value_counts())

Training data: (800, 10)
Testing data: (200, 10)

Training target distribution:
Optimization_Category
Low Efficiency         520
Moderate Efficiency    146
High Efficiency        134
Name: count, dtype: int64

Testing target distribution:
Optimization_Category
Low Efficiency         130
Moderate Efficiency     37
High Efficiency         33
Name: count, dtype: int64


In [23]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

efficiency_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight="balanced"
            )
        )
    ]
)

efficiency_model.fit(X_train_e, y_train_e)

print("Efficiency model trained successfully!")

Efficiency model trained successfully!


In [24]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Predictions
y_pred_e = efficiency_model.predict(X_test_e)

# Metrics
accuracy = accuracy_score(y_test_e, y_pred_e)
balanced_acc = balanced_accuracy_score(y_test_e, y_pred_e)

print("Accuracy:", round(accuracy * 100, 2), "%")
print("Balanced Accuracy:", round(balanced_acc * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test_e, y_pred_e))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_e, y_pred_e))

Accuracy: 52.0 %
Balanced Accuracy: 49.25 %

Classification Report:
                     precision    recall  f1-score   support

    High Efficiency       0.30      0.36      0.33        33
     Low Efficiency       0.67      0.55      0.60       130
Moderate Efficiency       0.39      0.57      0.46        37

           accuracy                           0.52       200
          macro avg       0.45      0.49      0.46       200
       weighted avg       0.56      0.52      0.53       200


Confusion Matrix:
[[12 21  0]
 [26 71 33]
 [ 2 14 21]]


In [25]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles in current folder:")
print(os.listdir())

Current folder:
C:\Users\SHEETAL\factory-ai-app\data

Files in current folder:
['.ipynb_checkpoints', 'AI41 2020.xlsx', 'energy_analysis.ipynb', 'energy_model.pkl', 'energy_threshold.pkl', 'hybrid_manufacturing_categorical.csv', 'Machine Health.ipynb', 'machine_health_model.pkl', 'Production Insight.ipynb', 'secom (1).zip', 'Steel_industry_data.csv']


In [26]:
import zipfile
import os

zip_path = "secom (1).zip"

with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("secom_dataset")

print("SECOM dataset extracted successfully!")

print("\nExtracted files:")
print(os.listdir("secom_dataset"))

SECOM dataset extracted successfully!

Extracted files:
['secom.data', 'secom.names', 'secom_labels.data']


In [27]:
import os

print(os.listdir("secom_dataset"))

['secom.data', 'secom.names', 'secom_labels.data']


In [28]:
import pandas as pd

# Load SECOM process measurements
secom_X = pd.read_csv(
    "secom_dataset/secom.data",
    sep=r"\s+",
    header=None
)

# Load SECOM labels
secom_y = pd.read_csv(
    "secom_dataset/secom_labels.data",
    sep=r"\s+",
    header=None
)

print("Features shape:", secom_X.shape)
print("Labels shape:", secom_y.shape)

print("\nTarget distribution:")
print(secom_y[0].value_counts())

print("\nFirst 5 feature rows:")
display(secom_X.head())

Features shape: (1567, 590)
Labels shape: (1567, 2)

Target distribution:
0
-1    1463
 1     104
Name: count, dtype: int64

First 5 feature rows:


,0,1,2,3,4,5,6,7,8,9,...,580,581,582,583,584,585,586,587,588,589
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,NaN,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,0.0060,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,0.0148,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,0.0044,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,NaN,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432


In [29]:
# Check missing values in SECOM features

missing_count = secom_X.isnull().sum()

print("Total missing values:", missing_count.sum())

print("Features with missing values:", (missing_count > 0).sum())

print("Features with more than 50% missing:",
      (missing_count > len(secom_X) * 0.50).sum())

print("\nTop 20 features with most missing values:")
print(missing_count.sort_values(ascending=False).head(20))

Total missing values: 41951
Features with missing values: 538
Features with more than 50% missing: 28

Top 20 features with most missing values:
292    1429
293    1429
158    1429
157    1429
492    1341
85     1341
358    1341
220    1341
244    1018
517    1018
109    1018
246    1018
383    1018
382    1018
384    1018
245    1018
111    1018
110    1018
516    1018
518    1018
dtype: int64


In [30]:
# Remove features with more than 50% missing values

missing_ratio = secom_X.isnull().mean()

secom_X_clean = secom_X.loc[
    :,
    missing_ratio <= 0.50
]

print("Original features:", secom_X.shape[1])
print("Features after removal:", secom_X_clean.shape[1])
print("Features removed:", secom_X.shape[1] - secom_X_clean.shape[1])

print("\nRemaining missing values:",
      secom_X_clean.isnull().sum().sum())


Original features: 590
Features after removal: 562
Features removed: 28

Remaining missing values: 11683


In [31]:
from sklearn.impute import SimpleImputer

# Median imputation for remaining missing values
imputer = SimpleImputer(strategy="median")

secom_X_imputed = pd.DataFrame(
    imputer.fit_transform(secom_X_clean),
    columns=secom_X_clean.columns
)

print("Shape after imputation:", secom_X_imputed.shape)
print(
    "Remaining missing values:",
    secom_X_imputed.isnull().sum().sum()
)

Shape after imputation: (1567, 562)
Remaining missing values: 0


In [32]:
# Prepare target variable

secom_y_binary = secom_y[0].map({
    -1: 0,   # Pass
     1: 1    # Fail
})

print("Target distribution:")
print(secom_y_binary.value_counts())

print("\nTarget percentages:")
print(
    (secom_y_binary.value_counts(normalize=True) * 100)
    .round(2)
)

Target distribution:
0
0    1463
1     104
Name: count, dtype: int64

Target percentages:
0
0    93.36
1     6.64
Name: proportion, dtype: float64


In [33]:
from sklearn.model_selection import train_test_split

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    secom_X_imputed,
    secom_y_binary,
    test_size=0.20,
    random_state=42,
    stratify=secom_y_binary
)

print("Training data:", X_train_s.shape)
print("Testing data:", X_test_s.shape)

print("\nTraining target distribution:")
print(y_train_s.value_counts())

print("\nTesting target distribution:")
print(y_test_s.value_counts())

Training data: (1253, 562)
Testing data: (314, 562)

Training target distribution:
0
0    1170
1      83
Name: count, dtype: int64

Testing target distribution:
0
0    293
1     21
Name: count, dtype: int64


In [34]:
from sklearn.ensemble import RandomForestClassifier

secom_model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

secom_model.fit(X_train_s, y_train_s)

print("SECOM model trained successfully!")

SECOM model trained successfully!


In [35]:
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix
)

# Predictions
y_pred_s = secom_model.predict(X_test_s)

# Metrics
accuracy_s = accuracy_score(y_test_s, y_pred_s)
balanced_acc_s = balanced_accuracy_score(y_test_s, y_pred_s)

print("Accuracy:", round(accuracy_s * 100, 2), "%")
print("Balanced Accuracy:", round(balanced_acc_s * 100, 2), "%")

print("\nClassification Report:")
print(
    classification_report(
        y_test_s,
        y_pred_s,
        target_names=["Pass", "Fail"]
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_s, y_pred_s))

Accuracy: 93.31 %
Balanced Accuracy: 50.0 %

Classification Report:
              precision    recall  f1-score   support

        Pass       0.93      1.00      0.97       293
        Fail       0.00      0.00      0.00        21

    accuracy                           0.93       314
   macro avg       0.47      0.50      0.48       314
weighted avg       0.87      0.93      0.90       314


Confusion Matrix:
[[293   0]
 [ 21   0]]


C:\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Python314\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [36]:
from sklearn.linear_model import LogisticRegression

secom_lr = LogisticRegression(
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

secom_lr.fit(X_train_s, y_train_s)

print("Logistic Regression model trained successfully!")

Logistic Regression model trained successfully!


C:\Python314\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 2000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=2000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [37]:
# Logistic Regression predictions
y_pred_lr = secom_lr.predict(X_test_s)

accuracy_lr = accuracy_score(y_test_s, y_pred_lr)
balanced_acc_lr = balanced_accuracy_score(y_test_s, y_pred_lr)

print("Accuracy:", round(accuracy_lr * 100, 2), "%")
print("Balanced Accuracy:", round(balanced_acc_lr * 100, 2), "%")

print("\nClassification Report:")
print(
    classification_report(
        y_test_s,
        y_pred_lr,
        target_names=["Pass", "Fail"],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_s, y_pred_lr))

Accuracy: 77.07 %
Balanced Accuracy: 63.4 %

Classification Report:
              precision    recall  f1-score   support

        Pass       0.95      0.79      0.87       293
        Fail       0.14      0.48      0.22        21

    accuracy                           0.77       314
   macro avg       0.55      0.63      0.54       314
weighted avg       0.90      0.77      0.82       314


Confusion Matrix:
[[232  61]
 [ 11  10]]


In [38]:
from sklearn.feature_selection import SelectKBest, f_classif

# Select top 100 features using training data only
selector = SelectKBest(
    score_func=f_classif,
    k=100
)

X_train_selected = selector.fit_transform(
    X_train_s,
    y_train_s
)

X_test_selected = selector.transform(
    X_test_s
)

print("Original features:", X_train_s.shape[1])
print("Selected features:", X_train_selected.shape[1])
print("Training shape:", X_train_selected.shape)
print("Testing shape:", X_test_selected.shape)

Original features: 562
Selected features: 100
Training shape: (1253, 100)
Testing shape: (314, 100)


C:\Python314\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:110: UserWarning: Features [  5  13  42  49  52  69  94 135 143 170 171 178 181 182 183 184 185 186
 217 220 221 222 223 224 225 226 227 228 231 232 233 234 244 245 246 247
 248 249 250 251 252 253 254 264 272 299 300 301 308 311 312 313 314 315
 316 347 352 353 354 355 356 357 358 361 362 363 364 374 375 376 377 378
 379 380 381 382 383 384 394 402 429 430 431 438 441 442 443 444 445 446
 461 477 480 481 482 483 484 485 486 487 488 491 492 493 494 504 505 506
 507 508 509 510 511 512 513 514] are constant.
  warnings.warn("Features %s are constant." % constant_features_idx, UserWarning)
C:\Python314\Lib\site-packages\sklearn\feature_selection\_univariate_selection.py:111: RuntimeWarning: invalid value encountered in divide
  f = msb / msw


In [39]:
from sklearn.ensemble import HistGradientBoostingClassifier

secom_gb = HistGradientBoostingClassifier(
    max_iter=300,
    learning_rate=0.05,
    max_leaf_nodes=15,
    l2_regularization=1.0,
    random_state=42
)

secom_gb.fit(
    X_train_selected,
    y_train_s
)

print("Gradient Boosting model trained successfully!")

Gradient Boosting model trained successfully!


In [40]:
# Gradient Boosting predictions
y_pred_gb = secom_gb.predict(X_test_selected)

accuracy_gb = accuracy_score(y_test_s, y_pred_gb)
balanced_acc_gb = balanced_accuracy_score(y_test_s, y_pred_gb)

print("Accuracy:", round(accuracy_gb * 100, 2), "%")
print("Balanced Accuracy:", round(balanced_acc_gb * 100, 2), "%")

print("\nClassification Report:")
print(
    classification_report(
        y_test_s,
        y_pred_gb,
        target_names=["Pass", "Fail"],
        zero_division=0
    )
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test_s, y_pred_gb))

Accuracy: 93.31 %
Balanced Accuracy: 50.0 %

Classification Report:
              precision    recall  f1-score   support

        Pass       0.93      1.00      0.97       293
        Fail       0.00      0.00      0.00        21

    accuracy                           0.93       314
   macro avg       0.47      0.50      0.48       314
weighted avg       0.87      0.93      0.90       314


Confusion Matrix:
[[293   0]
 [ 21   0]]


In [42]:
import pandas as pd

df = pd.read_csv("smart_manufacturing_dataset.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Shape: (10000, 9)

Columns:
['Timestamp', 'Machine ID', 'Material Category', 'Material Name', 'Quantity Used (kg)', 'Recycled Material (%)', 'Energy Consumption (kWh)', 'Production Output (Units)', 'Defect Rate (%)']

Data Types:
Timestamp                        str
Machine ID                       str
Material Category                str
Material Name                    str
Quantity Used (kg)           float64
Recycled Material (%)        float64
Energy Consumption (kWh)     float64
Production Output (Units)      int64
Defect Rate (%)              float64
dtype: object

Missing Values:
Timestamp                    0
Machine ID                   0
Material Category            0
Material Name                0
Quantity Used (kg)           0
Recycled Material (%)        0
Energy Consumption (kWh)     0
Production Output (Units)    0
Defect Rate (%)              0
dtype: int64


In [43]:
print("=== NUMERICAL SUMMARY ===")
print(df.describe())

print("\n=== UNIQUE VALUES ===")
for col in ["Machine ID", "Material Category", "Material Name"]:
    print(f"\n{col}:")
    print(df[col].nunique())
    print(df[col].unique()[:20])

=== NUMERICAL SUMMARY ===
       Quantity Used (kg)  Recycled Material (%)  Energy Consumption (kWh)  \
count        10000.000000           10000.000000              10000.000000   
mean           124.534008               4.060385                140.111901   
std             43.136723              10.908877                 34.540921   
min             50.010000               0.000000                 80.010000   
25%             87.215000               0.000000                110.647500   
50%            124.250000               0.000000                140.315000   
75%            161.752500               0.000000                169.877500   
max            200.000000              49.960000                200.000000   

       Production Output (Units)  Defect Rate (%)  
count               10000.000000     10000.000000  
mean                  553.481500         2.526966  
std                   260.877808         1.442364  
min                   100.000000         0.000000  
25%        

In [44]:
# Features used to predict energy consumption
X = df[
    [
        "Machine ID",
        "Material Category",
        "Material Name",
        "Quantity Used (kg)",
        "Recycled Material (%)",
        "Production Output (Units)",
        "Defect Rate (%)"
    ]
]

# Target
y = df["Energy Consumption (kWh)"]

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFeatures:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

X shape: (10000, 7)
y shape: (10000,)

Features:
['Machine ID', 'Material Category', 'Material Name', 'Quantity Used (kg)', 'Recycled Material (%)', 'Production Output (Units)', 'Defect Rate (%)']

Target:
Energy Consumption (kWh)


In [45]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining target:", y_train.shape)
print("Testing target:", y_test.shape)

Training data: (8000, 7)
Testing data: (2000, 7)

Training target: (8000,)
Testing target: (2000,)


In [46]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical_features = [
    "Machine ID",
    "Material Category",
    "Material Name"
]

numerical_features = [
    "Quantity Used (kg)",
    "Recycled Material (%)",
    "Production Output (Units)",
    "Defect Rate (%)"
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        ),
        (
            "num",
            "passthrough",
            numerical_features
        )
    ]
)

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)
print("Preprocessor ready.")

Categorical features: ['Machine ID', 'Material Category', 'Material Name']
Numerical features: ['Quantity Used (kg)', 'Recycled Material (%)', 'Production Output (Units)', 'Defect Rate (%)']
Preprocessor ready.


In [47]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

energy_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

energy_model.fit(X_train, y_train)

print("Energy prediction model trained successfully.")

Energy prediction model trained successfully.


In [48]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = energy_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== ENERGY PREDICTION RESULTS ===")
print(f"MAE  : {mae:.2f} kWh")
print(f"RMSE : {rmse:.2f} kWh")
print(f"R²   : {r2:.4f}")


=== ENERGY PREDICTION RESULTS ===
MAE  : 30.55 kWh
RMSE : 35.44 kWh
R²   : -0.0461


In [49]:
correlation = df[
    [
        "Quantity Used (kg)",
        "Recycled Material (%)",
        "Energy Consumption (kWh)",
        "Production Output (Units)",
        "Defect Rate (%)"
    ]
].corr()

print(correlation["Energy Consumption (kWh)"].sort_values(ascending=False))

Energy Consumption (kWh)     1.000000
Quantity Used (kg)           0.004616
Production Output (Units)   -0.001661
Recycled Material (%)       -0.009729
Defect Rate (%)             -0.010902
Name: Energy Consumption (kWh), dtype: float64


In [50]:
print("=== ENERGY BY MACHINE ===")
print(
    df.groupby("Machine ID")["Energy Consumption (kWh)"]
    .agg(["mean", "std", "min", "max"])
    .round(2)
)

print("\n=== ENERGY BY MATERIAL CATEGORY ===")
print(
    df.groupby("Material Category")["Energy Consumption (kWh)"]
    .agg(["mean", "std", "min", "max"])
    .round(2)
)

print("\n=== ENERGY BY MATERIAL NAME ===")
print(
    df.groupby("Material Name")["Energy Consumption (kWh)"]
    .agg(["mean", "std", "min", "max"])
    .round(2)
)

=== ENERGY BY MACHINE ===
              mean    std    min     max
Machine ID                              
M001        139.42  34.15  80.09  199.94
M002        139.68  34.88  80.01  199.99
M003        141.38  34.31  80.01  199.91
M004        139.34  35.12  80.01  200.00
M005        143.02  34.26  80.22  199.88
M006        139.06  33.86  80.08  200.00
M007        141.03  34.18  80.08  199.94
M008        137.66  34.87  80.07  199.91
M009        139.30  35.10  80.01  199.81
M010        141.18  34.54  80.24  199.72

=== ENERGY BY MATERIAL CATEGORY ===
                         mean    std    min     max
Material Category                                  
Finished Material      140.21  34.27  80.01  200.00
Hazardous Material     139.83  34.70  80.08  199.99
Intermediate Material  140.26  34.97  80.01  199.71
Process Additive       142.15  34.71  80.03  200.00
Raw Material           139.22  33.90  80.01  199.88
Recycled Material      139.12  34.71  80.02  199.93

=== ENERGY BY MATERIAL NAME 

In [51]:
df["Energy per Unit"] = (
    df["Energy Consumption (kWh)"] /
    df["Production Output (Units)"]
)

print("=== ENERGY PER UNIT ===")
print(df["Energy per Unit"].describe().round(4))

print("\n=== BEST 10 ENERGY-EFFICIENT ROWS ===")
print(
    df[
        [
            "Machine ID",
            "Material Name",
            "Energy Consumption (kWh)",
            "Production Output (Units)",
            "Energy per Unit",
            "Defect Rate (%)"
        ]
    ]
    .sort_values("Energy per Unit")
    .head(10)
)

=== ENERGY PER UNIT ===
count    10000.0000
mean         0.3565
std          0.2833
min          0.0814
25%          0.1770
50%          0.2530
75%          0.4245
max          1.9595
Name: Energy per Unit, dtype: float64

=== BEST 10 ENERGY-EFFICIENT ROWS ===
     Machine ID     Material Name  Energy Consumption (kWh)  \
6674       M003           Solvent                     80.24   
2273       M003  Lead-Based Alloy                     81.05   
2743       M004    Industrial Oil                     81.42   
1358       M010  Lead-Based Alloy                     81.10   
6022       M001       Sheet Metal                     82.13   
5278       M003     Circuit Board                     80.48   
8044       M005            Cotton                     80.45   
6599       M008     Circuit Board                     82.41   
1866       M008       Sheet Metal                     80.40   
3104       M002          Textiles                     80.73   

      Production Output (Units)  Energy per U

In [52]:
import pandas as pd

df = pd.read_csv("Intelligent_Production_IIoT.csv")

print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nFirst 5 Rows:")
display(df.head())

Shape: (15000, 45)

Columns:
['record_id', 'machine_id', 'production_line', 'batch_id', 'shift', 'product_type', 'ambient_temperature', 'humidity', 'air_pressure', 'ambient_vibration', 'motor_temperature', 'spindle_speed', 'motor_current', 'torque', 'vibration', 'acoustic_level', 'bearing_temperature', 'process_temperature', 'hydraulic_pressure', 'flow_rate', 'coolant_temperature', 'material_feed_rate', 'feed_pressure', 'cycle_time', 'throughput_rate', 'production_volume', 'machine_utilization', 'resource_utilization', 'operator_load', 'power_consumption', 'energy_per_unit', 'defect_rate', 'quality_score', 'first_pass_yield', 'fault_event_count', 'downtime_minutes', 'maintenance_frequency', 'production_cost_per_unit', 'resource_efficiency', 'production_efficiency', 'energy_saving_pct', 'downtime_reduction_pct', 'cost_reduction_pct', 'benefit_score', 'operation_mode']

Data Types:
record_id                     int64
machine_id                      str
production_line                 str

,record_id,machine_id,production_line,batch_id,shift,product_type,ambient_temperature,humidity,air_pressure,ambient_vibration,...,downtime_minutes,maintenance_frequency,production_cost_per_unit,resource_efficiency,production_efficiency,energy_saving_pct,downtime_reduction_pct,cost_reduction_pct,benefit_score,operation_mode
0,1,M07,Line_A,B0343,Morning,Product_C,24.657935,69.272037,102.335679,0.366190,...,25.081059,2.172029,8.028928,99.0,99.0,9.364688,49.353148,26.199425,68.260235,High_Throughput
1,2,M20,Line_E,B0593,Afternoon,Product_D,15.953337,59.927354,100.055317,0.472410,...,20.400191,2.576074,8.537013,99.0,99.0,3.129223,56.434784,30.016525,68.491891,High_Throughput
2,3,M15,Line_E,B0205,Afternoon,Product_C,21.761215,44.487079,102.794636,0.478138,...,11.608597,2.357031,7.372440,99.0,99.0,6.433856,61.539859,33.412663,68.837230,Balanced_Production
3,4,M11,Line_B,B0993,Morning,Product_A,21.973528,59.022870,98.258859,0.417760,...,25.542778,2.840125,8.622625,99.0,99.0,3.284783,43.948708,34.699694,67.074754,Balanced_Production
4,5,M08,Line_E,B0961,Night,Product_C,20.595578,54.665654,99.217605,0.283555,...,19.759181,3.410520,7.521289,99.0,99.0,12.958237,52.025084,31.553079,69.084109,Precision_Quality


In [53]:
target = "power_consumption"

numeric_cols = df.select_dtypes(include="number").columns

correlation = (
    df[numeric_cols]
    .corr()[target]
    .sort_values(ascending=False)
)

print("=== CORRELATION WITH POWER CONSUMPTION ===")
print(correlation)

print("\n=== TARGET SUMMARY ===")
print(df[target].describe())

print("\n=== POTENTIAL LEAKAGE COLUMNS ===")
print([
    col for col in df.columns
    if any(word in col.lower() for word in [
        "energy", "saving", "efficiency", "benefit"
    ])
])

=== CORRELATION WITH POWER CONSUMPTION ===
power_consumption           1.000000
spindle_speed               0.869248
throughput_rate             0.741358
flow_rate                   0.733615
motor_current               0.554495
production_volume           0.446521
acoustic_level              0.333707
energy_per_unit             0.282608
production_efficiency       0.177893
production_cost_per_unit    0.160502
torque                      0.137197
vibration                   0.068161
motor_temperature           0.019413
downtime_minutes            0.018216
defect_rate                 0.015699
record_id                   0.012822
bearing_temperature         0.012558
machine_utilization         0.012448
fault_event_count           0.011238
hydraulic_pressure          0.011135
ambient_temperature         0.010391
resource_utilization        0.009274
maintenance_frequency       0.009090
humidity                    0.006167
operator_load               0.002100
material_feed_rate         -0.00

In [54]:
# Check mathematical relationships with power_consumption

check_cols = [
    "power_consumption",
    "spindle_speed",
    "throughput_rate",
    "flow_rate",
    "motor_current",
    "production_volume",
    "cycle_time",
    "torque"
]

print(df[check_cols].describe().T)

print("\n=== SAMPLE VALUES ===")
print(df[check_cols].head(10).to_string(index=False))

                     count         mean         std          min          25%  \
power_consumption  15000.0    19.784479    1.731123    13.650633    18.609828   
spindle_speed      14642.0  2850.524255  423.660252  1500.000000  2559.811083   
throughput_rate    15000.0   113.549796    8.075672    83.437338   108.094474   
flow_rate          14642.0    67.797916    5.298359    46.264863    64.174242   
motor_current      14616.0    24.551220    2.790231    13.720123    22.652374   
production_volume  15000.0   111.726879   13.428791    74.357002   101.626838   
cycle_time         15000.0    32.099246    3.815783    20.000000    29.535842   
torque             14613.0    53.090018    5.125547    32.059558    49.620896   

                           50%          75%          max  
power_consumption    19.779331    20.961176    25.000000  
spindle_speed      2851.862022  3135.629252  4000.000000  
throughput_rate     113.550883   119.083805   135.000000  
flow_rate            68.145201    

In [55]:
features = [
    "spindle_speed",
    "throughput_rate",
    "flow_rate",
    "motor_current",
    "production_volume",
    "cycle_time",
    "torque",
    "vibration",
    "acoustic_level",
    "machine_utilization",
    "resource_utilization",
    "material_feed_rate",
    "feed_pressure",
    "hydraulic_pressure",
    "motor_temperature",
    "bearing_temperature",
    "process_temperature",
    "coolant_temperature",
    "ambient_temperature",
    "humidity",
    "air_pressure"
]

target = "power_consumption"

In [56]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

features = [
    "spindle_speed",
    "throughput_rate",
    "flow_rate",
    "motor_current",
    "production_volume",
    "cycle_time",
    "torque",
    "vibration",
    "acoustic_level",
    "machine_utilization",
    "resource_utilization",
    "material_feed_rate",
    "feed_pressure",
    "hydraulic_pressure",
    "motor_temperature",
    "bearing_temperature",
    "process_temperature",
    "coolant_temperature",
    "ambient_temperature",
    "humidity",
    "air_pressure"
]

target = "power_consumption"

X = df[features]
y = df[target]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

# Numerical preprocessing
numeric_features = X.select_dtypes(include="number").columns

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features)
])

# Random Forest model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

# Train
model.fit(X_train, y_train)

# Predict
y_pred = model.predict(X_test)

# Evaluation
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== PRODUCTION INSIGHT MODEL ===")
print(f"MAE  : {mae:.3f} kWh")
print(f"RMSE : {rmse:.3f} kWh")
print(f"R²   : {r2:.4f}")

=== PRODUCTION INSIGHT MODEL ===
MAE  : 0.671 kWh
RMSE : 0.841 kWh
R²   : 0.7630


In [57]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

baseline_mae = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2 = r2_score(y_test, baseline_pred)

print("=== BASELINE MODEL ===")
print(f"MAE  : {baseline_mae:.3f} kWh")
print(f"RMSE : {baseline_rmse:.3f} kWh")
print(f"R²   : {baseline_r2:.4f}")

print("\n=== RANDOM FOREST ===")
print(f"MAE  : {mae:.3f} kWh")
print(f"RMSE : {rmse:.3f} kWh")
print(f"R²   : {r2:.4f}")

=== BASELINE MODEL ===
MAE  : 1.385 kWh
RMSE : 1.728 kWh
R²   : -0.0005

=== RANDOM FOREST ===
MAE  : 0.671 kWh
RMSE : 0.841 kWh
R²   : 0.7630


In [58]:
import joblib
import os

os.makedirs("models", exist_ok=True)

joblib.dump(model, "models/production_energy_model.pkl")

print("✅ Production Energy Model saved successfully.")
print("Path: models/production_energy_model.pkl")

✅ Production Energy Model saved successfully.
Path: models/production_energy_model.pkl


In [59]:
import os

print("Current notebook folder:")
print(os.getcwd())

print("\nFiles/folders here:")
print(os.listdir())

Current notebook folder:
C:\Users\SHEETAL\factory-ai-app\data

Files/folders here:
['.ipynb_checkpoints', 'AI41 2020.xlsx', 'energy_analysis.ipynb', 'energy_model.pkl', 'energy_threshold.pkl', 'hybrid_manufacturing_categorical.csv', 'Intelligent_Production_IIoT.csv', 'Machine Health.ipynb', 'machine_health_model.pkl', 'models', 'Production Insight.ipynb', 'secom_dataset', 'smart_manufacturing_dataset.csv', 'Steel_industry_data.csv']


In [60]:
import joblib
import os

model_path = os.path.join(
    os.getcwd(),
    "models",
    "production_energy_model.pkl"
)

joblib.dump(model, model_path)

print("✅ Model saved!")
print("Full path:")
print(model_path)

print("\nFile exists:", os.path.exists(model_path))
print("File size:", os.path.getsize(model_path), "bytes")

✅ Model saved!
Full path:
C:\Users\SHEETAL\factory-ai-app\data\models\production_energy_model.pkl

File exists: True
File size: 218487747 bytes
